In [ ]:
### this notebook sets up all the directories and the required files, run this everytime you open jupyter

In [8]:
import os
import numpy as np
import glob


## MASTER CONFIGURATION - Run this every time you open Jupyter!


# Absolute paths - never change these
BASE_DIR   = r'C:\Users\srika\gw_wd'
EVENTS_DIR = os.path.join(BASE_DIR, 'data', 'events')
NOISE_DIR  = os.path.join(BASE_DIR, 'data', 'noise')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
RESULTS_DIR = os.path.join(BASE_DIR, 'results', 'figures')

print("PROJECT LOADER")

print(f"\n📍 Base    : {BASE_DIR}")
print(f"📍 Events  : {EVENTS_DIR}")
print(f"📍 Noise   : {NOISE_DIR}")
print(f"📍 Models  : {MODELS_DIR}")
print(f"📍 Results : {RESULTS_DIR}")

# Check everything exists
all_good = True
for path in [EVENTS_DIR, NOISE_DIR]:
    if os.path.exists(path):
        print(f"\n EXISTS: {path}")
    else:
        print(f"\n MISSING: {path}")
        all_good = False

print(f"\n{' All paths OK!' if all_good else ' Some paths missing!'}")

PROJECT LOADER

📍 Base    : C:\Users\srika\gw_wd
📍 Events  : C:\Users\srika\gw_wd\data\events
📍 Noise   : C:\Users\srika\gw_wd\data\noise
📍 Models  : C:\Users\srika\gw_wd\models
📍 Results : C:\Users\srika\gw_wd\results\figures

 EXISTS: C:\Users\srika\gw_wd\data\events

 EXISTS: C:\Users\srika\gw_wd\data\noise

 All paths OK!


In [9]:
# Check all data files
event_files = sorted([f for f in glob.glob(os.path.join(EVENTS_DIR, '*.npy'))
                      if 'meta' not in f])
noise_files = sorted([f for f in glob.glob(os.path.join(NOISE_DIR, '*.npy'))
                      if 'meta' not in f])


print("DATA FILE CHECK")


print(f"\n Event files : {len(event_files)}")
for f in event_files:
    size = os.path.getsize(f) / 1024 / 1024
    print(f"  {os.path.basename(f):35s} {size:.2f} MB")

print(f"\n Noise files : {len(noise_files)}")
for f in noise_files[:5]:
    size = os.path.getsize(f) / 1024 / 1024
    print(f"    {os.path.basename(f):35s} {size:.2f} MB")
if len(noise_files) > 5:
    print(f"    and {len(noise_files)-5} more files")

total_size = sum(os.path.getsize(f) for f in event_files + noise_files) / 1024 / 1024
print(f"\n Total size: {total_size:.1f} MB")

# Set flag for other notebooks
DATA_AVAILABLE = len(event_files) > 0 and len(noise_files) > 0
print(f"\n{' Data ready!' if DATA_AVAILABLE else ' Need to re-download!'}")


DATA FILE CHECK

 Event files : 8
  GW150914_H1.npy                     1.00 MB
  GW150914_L1.npy                     1.00 MB
  GW151226_H1.npy                     1.00 MB
  GW151226_L1.npy                     1.00 MB
  GW170814_H1.npy                     1.00 MB
  GW170814_L1.npy                     1.00 MB
  GW170817_H1.npy                     1.00 MB
  GW170817_L1.npy                     1.00 MB

 Noise files : 14
    noise_001_H1.npy                    1.00 MB
    noise_001_L1.npy                    1.00 MB
    noise_002_H1.npy                    1.00 MB
    noise_002_L1.npy                    1.00 MB
    noise_003_H1.npy                    1.00 MB
    and 9 more files

 Total size: 22.0 MB

 Data ready!


In [10]:
def load_all_events():
    """Load all gravitational wave events into memory"""
    
    events = {}
    event_names = sorted(list(set([
        os.path.basename(f).split('_')[0] 
        for f in event_files
    ])))
    
    for event_name in event_names:
        events[event_name] = {}
        for detector in ['H1', 'L1']:
            filepath = os.path.join(EVENTS_DIR, f'{event_name}_{detector}.npy')
            metapath = os.path.join(EVENTS_DIR, f'{event_name}_{detector}_meta.npy')
            
            if os.path.exists(filepath):
                events[event_name][detector] = {
                    'data': np.load(filepath),
                    'meta': np.load(metapath, allow_pickle=True).item()
                }
    
    return events


def load_all_noise():
    """Load all noise segments into memory"""
    
    noise = []
    for i in range(len(noise_files) // 2):  # Divide by 2 for H1 and L1
        segment = {}
        for detector in ['H1', 'L1']:
            filepath = os.path.join(NOISE_DIR, f'noise_{i:03d}_{detector}.npy')
            metapath = os.path.join(NOISE_DIR, f'noise_{i:03d}_{detector}_meta.npy')
            
            if os.path.exists(filepath):
                segment[detector] = {
                    'data': np.load(filepath),
                    'meta': np.load(metapath, allow_pickle=True).item()
                }
        
        if segment:
            noise.append(segment)
    
    return noise


# Load everything
print("Loading data into memory...")


events = load_all_events()
noise_segments = load_all_noise()
print(f"\n Loaded {len(events)} events:")
for name, detectors in events.items():
    for det, info in detectors.items():
        samples = len(info['data'])
        sr = info['meta']['sample_rate']
        print(f"   • {name} {det}: {samples:,} samples @ {sr} Hz")

print(f"\n Loaded {len(noise_segments)} noise segments")

print(" EVERYTHING LOADED! Ready to work")

print("\nAvailable variables:")
print("  • events          → all GW event data")
print("  • noise_segments  → all noise data")
print("  • EVENTS_DIR      → path to events folder")
print("  • NOISE_DIR       → path to noise folder") 

Loading data into memory...

 Loaded 4 events:
   • GW150914 H1: 131,072 samples @ 4096.0 Hz
   • GW150914 L1: 131,072 samples @ 4096.0 Hz
   • GW151226 H1: 131,072 samples @ 4096.0 Hz
   • GW151226 L1: 131,072 samples @ 4096.0 Hz
   • GW170814 H1: 131,072 samples @ 4096.0 Hz
   • GW170814 L1: 131,072 samples @ 4096.0 Hz
   • GW170817 H1: 131,072 samples @ 4096.0 Hz
   • GW170817 L1: 131,072 samples @ 4096.0 Hz

 Loaded 5 noise segments
 EVERYTHING LOADED! Ready to work

Available variables:
  • events          → all GW event data
  • noise_segments  → all noise data
  • EVENTS_DIR      → path to events folder
  • NOISE_DIR       → path to noise folder
